# NB05：蛋白質分析（凱氏定氮法, Kjeldahl Method）

**食品分析實驗 週次 5｜Nielsen's Food Analysis, 6th ed. 對應章節：蛋白質 (Protein, Ch.18)**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用凱氏法公式，從滴定體積計算樣品的 **%N** 與**蛋白質%**（換算係數 5.7 / 6.25 / 6.38）。
2. 說出「換算係數選錯」造成的**系統誤差(systematic error)**量級（小麥係數 5.7 vs 通用值 6.25）。
3. 計算蒸餾回收標準品（硫酸銨）的**回收率(recovery)**，判斷蒸餾/捕氨是否完整。
4. 彙整全班盲樣（全脂奶粉，n=5，每組 1 管）的平均值 ± 標準差，體會「班級數據」比單組數據更可信的原因。
5. 重新計算**三聚氰胺(melamine)事件**：摻偽多少比例會虛增多少表觀蛋白質。
6. 用**靈敏度分析(sensitivity analysis)**說明滴定讀取誤差如何隨樣品重量放大或縮小。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**。
- `load_data()` 支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb05_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，把網址傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的 `google.colab.files.upload()` 程式碼。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，不是真實量測值。做作業時請換成你們班的實際數據。

## 重要背景：本週的 8 管限制

消化爐＋蒸餾儀**一次只能容納 8 管**，因此本週的分工是：

| 管別 | 數量 | 說明 |
|---|---|---|
| 全脂奶粉盲樣 | 5 管 | 每組消化 1 管，全班湊成 n=5 |
| 試劑空白 (reagent blank) | 1 管 | 全班共用，不加樣品 |
| 硫酸銨回收標準品 | 1 管 | QC 用，理論 %N = 21.20% |
| 高筋麵粉 | 1 管 | 由 1 組代表操作，結果全班共用 |
| **合計** | **8 管** | 剛好用滿設備容量 |

低筋（蛋糕）麵粉**沒有現場加測**（排程已滿 8 管），其數值由**助教提供**（前次操作數據），只用來跟高筋麵粉比較係數與蛋白質差異。

## 資料格式 (Data Schema)

延續 NB01 的長格式（long format），本週多加一欄 `weight_g`（樣品重量），因為凱氏法的計算同時需要「滴定體積」與「樣品重量」：

| 欄位 | 說明 |
|---|---|
| `group` | 組別（1–5）；`0` 代表全班共用或 TA 提供，非特定分組 |
| `student` | 代號（例：`G1` 代表第 1 組；`shared`／`TA` 代表全班共用或助教提供）|
| `item` | 管別代號：`milk_powder_blind`、`reagent_blank`、`ammonium_sulfate_std`、`high_gluten_flour`、`low_gluten_flour` |
| `rep` | 重複次數（本週每管只滴定 1 次，固定為 1）|
| `value` | 滴定用去的 HCl 體積（mL）|
| `unit` | `value` 的單位（本週固定為 `mL`）|
| `temp_C` | 本週不適用（凱氏法計算不需要溫度），保留欄位以與其他週次的長格式一致，留空 |
| `weight_g` | 樣品重量（g）；試劑空白無樣品，留空 |
| `note` | 備註欄，說明該管的角色/來源 |

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`。**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb05_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,weight_g,note
1,G1,milk_powder_blind,1,14.67,mL,,0.5005,5 組各消化 1 管盲樣（設備僅容納 8 管）
2,G2,milk_powder_blind,1,15.03,mL,,0.5011,5 組各消化 1 管盲樣（設備僅容納 8 管）
3,G3,milk_powder_blind,1,14.63,mL,,0.4971,5 組各消化 1 管盲樣（設備僅容納 8 管）
4,G4,milk_powder_blind,1,14.8,mL,,0.5002,5 組各消化 1 管盲樣（設備僅容納 8 管）
5,G5,milk_powder_blind,1,14.71,mL,,0.5,5 組各消化 1 管盲樣（設備僅容納 8 管）
0,shared,reagent_blank,1,0.31,mL,,,試劑空白，全班共用 1 管，不加樣品
0,shared,ammonium_sulfate_std,1,7.84,mL,,0.05,蒸餾回收標準品，理論 N%=21.20% (硫酸銨)
1,G1,high_gluten_flour,1,16.58,mL,,1.0,高筋麵粉，由 G1 代表操作，結果全班共用
0,TA,low_gluten_flour,1,10.32,mL,,1.0,TA 提供（前次操作數據），本次消化爐排程已滿 8 管，未現場加測
"""

In [ ]:
def load_data(source=None):
    """Load NB05 Kjeldahl data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, weight_g, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb05_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            df = pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")
            df = None
    else:
        local_path = os.path.join("data", "nb05_sample.csv")
        if os.path.exists(local_path):
            df = pd.read_csv(local_path)
        else:
            df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    if df is None:
        df = pd.read_csv(io.StringIO(SAMPLE_CSV))

    df["note"] = df["note"].fillna("")
    return df


df = load_data()
print(f"讀入 {len(df)} 筆資料（8 管 + 1 筆 TA 提供值），"
      f"涵蓋項目：{sorted(df['item'].unique())}")
df

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際量測值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. 核心公式：從滴定體積到蛋白質 %

$$\%N = \frac{(V_s - V_b)\times N_{HCl} \times 14.007}{W_{mg}} \times 100$$

$$\text{蛋白質}\% = \%N \times \text{換算係數}$$

其中 $V_s$ 為樣品滴定用去的 HCl 體積（mL）、$V_b$ 為試劑空白用去的體積（mL）、$N_{HCl}$ 為 HCl 的標準化濃度（eq/L）、$W_{mg}$ 為樣品重量（**毫克**）、14.007 為氮的原子量。

換算係數依樣品種類而不同（凱氏法量到的是「總氮」，不是「蛋白質」，換算係數是「假設氮全部來自蛋白質」下的近似值）：

| 換算係數 | 適用 | 理由(簡述) |
|---|---|---|
| 6.25 | 一般食品（通用值） | 假設蛋白質平均含氮 16%（1/0.16 = 6.25） |
| 5.7 | 小麥／穀類 | 穀類蛋白（麩質）含氮比例較高，需用較小係數 |
| 6.38 | 乳品 | 酪蛋白等乳蛋白含氮比例較低，需用較大係數 |

我們的 `df` 裡的 `weight_g` 是**公克**，公式需要**毫克**，記得換算（× 1000）。

In [ ]:
FACTORS = {"general": 6.25, "wheat": 5.7, "dairy": 6.38}


def pct_N(V_s, V_b, N_HCl, W_mg):
    """%N = (V_s - V_b) * N_HCl * 14.007 / W_mg * 100"""
    return (V_s - V_b) * N_HCl * 14.007 / W_mg * 100


def pct_protein(pctN_value, factor):
    return pctN_value * factor


N_HCL = 0.1000  # eq/L, standardized titrant (class-wide constant this week)

# Reagent blank (shared, 1 tube for the whole class)
V_BLANK = df.loc[df["item"] == "reagent_blank", "value"].iloc[0]
print(f"試劑空白 V_b = {V_BLANK:.2f} mL")

## 3. 全班盲樣：全脂奶粉 (n=5)

5 組各消化 1 管全脂奶粉盲樣，彙整成 **n=5** 的班級數據（每組貢獻一個值），計算平均值、標準差(SD)、95% CI，並用乳品係數 **6.38** 換算蛋白質%。

In [ ]:
milk = df[df["item"] == "milk_powder_blind"].copy()
milk["weight_mg"] = milk["weight_g"] * 1000
milk["pctN"] = pct_N(milk["value"], V_BLANK, N_HCL, milk["weight_mg"])
milk["protein_pct"] = pct_protein(milk["pctN"], FACTORS["dairy"])
milk[["group", "student", "value", "weight_g", "pctN", "protein_pct"]]

In [ ]:
n = len(milk)
mean_protein = milk["protein_pct"].mean()
sd_protein = milk["protein_pct"].std(ddof=1)
sem_protein = sd_protein / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
ci95_protein = t_crit * sem_protein

print(f"全脂奶粉盲樣 (n={n})：蛋白質% = {mean_protein:.2f} ± {ci95_protein:.2f} % (95% CI)")
print(f"SD = {sd_protein:.3f} %，RSD = {sd_protein / mean_protein * 100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(milk["student"], milk["protein_pct"], color="tab:blue", alpha=0.85)
ax.axhline(mean_protein, color="black", linestyle="--", linewidth=1,
           label=f"class mean = {mean_protein:.2f}%")
ax.axhspan(mean_protein - ci95_protein, mean_protein + ci95_protein,
           color="tab:orange", alpha=0.15, label="mean ± 95% CI")
ax.set_ylabel("Protein % (dairy factor 6.38)")
ax.set_xlabel("Group")
ax.set_title("Whole-milk-powder blind sample: n=5 (one tube per group)")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## 4. 高筋 vs 低筋麵粉：換算係數與選錯係數的代價

高筋麵粉本次由 1 組代表操作、結果全班共用；低筋（蛋糕）麵粉本次**沒有現場加測**，數值由助教提供。兩者都應該用小麥係數 **5.7**。

下面示範：如果誤用一般係數 6.25（而不是小麥係數 5.7）去換算高筋麵粉，蛋白質% 會被高估多少？

In [ ]:
hg = df[df["item"] == "high_gluten_flour"].iloc[0]
lg = df[df["item"] == "low_gluten_flour"].iloc[0]

hg_pctN = pct_N(hg["value"], V_BLANK, N_HCL, hg["weight_g"] * 1000)
lg_pctN = pct_N(lg["value"], V_BLANK, N_HCL, lg["weight_g"] * 1000)

hg_protein_correct = pct_protein(hg_pctN, FACTORS["wheat"])   # 5.7, correct
hg_protein_wrong = pct_protein(hg_pctN, FACTORS["general"])   # 6.25, WRONG factor
lg_protein = pct_protein(lg_pctN, FACTORS["wheat"])

overestimate_pct = (hg_protein_wrong - hg_protein_correct) / hg_protein_correct * 100

print(f"高筋麵粉：%N = {hg_pctN:.3f}%")
print(f"  用小麥係數 5.7（正確）  -> 蛋白質% = {hg_protein_correct:.2f}%")
print(f"  用一般係數 6.25（錯誤） -> 蛋白質% = {hg_protein_wrong:.2f}%")
print(f"  相對高估 = {overestimate_pct:.1f}%  <- 這是系統誤差(systematic error)，跟儀器準確度無關")
print()
print(f"低筋麵粉（TA 提供，未消化）：%N = {lg_pctN:.3f}%，蛋白質%(係數5.7) = {lg_protein:.2f}%")
print(f"高筋 - 低筋 蛋白質差 = {hg_protein_correct - lg_protein:.2f} 個百分點"
      "（這正是烘焙時「高筋做麵包、低筋做蛋糕」的依據）")

## 5. 品管：蒸餾回收標準品（硫酸銨）

用已知氮量的硫酸銨 $(NH_4)_2SO_4$ 走蒸餾＋滴定，檢查 NH₃ 有沒有被完全蒸出、捕捉。理論 %N（由分子量計算）：

$$\%N_{theory} = \frac{2 \times 14.007}{MW_{(NH_4)_2SO_4}} \times 100 \approx 21.20\%$$

回收率(recovery) = 實測 %N / 理論 %N × 100%，依方法可接受範圍通常需 **≥ 97%**。

In [ ]:
AS_N_THEORY = 2 * 14.007 / (2 * (14.007 + 4 * 1.008) + 32.06 + 4 * 16.00) * 100
print(f"硫酸銨理論 %N = {AS_N_THEORY:.2f}%")

asr = df[df["item"] == "ammonium_sulfate_std"].iloc[0]
asr_pctN = pct_N(asr["value"], V_BLANK, N_HCL, asr["weight_g"] * 1000)
asr_recovery = asr_pctN / AS_N_THEORY * 100

print(f"實測 %N = {asr_pctN:.2f}%")
print(f"回收率 = {asr_pctN:.2f} / {AS_N_THEORY:.2f} x 100 = {asr_recovery:.1f}%")
if asr_recovery >= 97:
    print("回收率 >= 97%，蒸餾/捕氨步驟判定為正常。")
else:
    print("回收率 < 97%，蒸餾/捕氨步驟可能有問題（NaOH 不足、蒸氣量不足、冷凝端漏氣等）。")

## 6. 三聚氰胺 (melamine) 事件重新計算

凱氏法（跟 Dumas 燃燒法一樣）量到的是「總氮」，無法分辨氮是蛋白質來的還是三聚氰胺 $(C_3H_6N_6)$ 來的。三聚氰胺分子量小、含氮量卻高達 **~66.6%**，是凱氏法檢驗上最著名的摻偽案例。

每混入 1 g 三聚氰胺 / 100 g 樣品（即 1% w/w），貢獻的 %N 增量、換算成表觀蛋白質(乳品係數 6.38)的增量分別是多少？

In [ ]:
MEL_MW = 3 * 12.011 + 6 * 1.008 + 6 * 14.007
MEL_N_PCT = 6 * 14.007 / MEL_MW * 100
apparent_protein_per_1pct = MEL_N_PCT / 100 * FACTORS["dairy"]

print(f"三聚氰胺 MW = {MEL_MW:.3f}, %N = {MEL_N_PCT:.2f}%")
print(f"每混入 1%% (w/w) 三聚氰胺 -> 虛增表觀蛋白質 ≈ {apparent_protein_per_1pct:.2f} 個百分點")
print(f"換句話說：每摻入 1 g 三聚氰胺，約虛增 {apparent_protein_per_1pct:.2f} g 的『表觀蛋白質』")

mel_rates = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0])
mel_apparent = mean_protein + mel_rates * apparent_protein_per_1pct

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mel_rates, mel_apparent, "o-", color="tab:red")
ax.set_xlabel("Melamine spiked into sample (% w/w)")
ax.set_ylabel("Apparent protein % (Kjeldahl, dairy factor 6.38)")
ax.set_title(f"Why Kjeldahl was fooled: milk powder baseline {mean_protein:.1f}% + melamine")
plt.tight_layout()
plt.show()

## 7. 靈敏度分析：滴定讀取誤差怎麼傳播到蛋白質 %

如果滴定管刻度判讀有 **±0.05 mL** 的誤差（人眼判讀滴定終點的常見誤差量級），對不同樣品的蛋白質% 影響有多大？

$$\Delta(\text{蛋白質}\%) = \Delta V \times N_{HCl} \times \frac{14.007}{W_{mg}} \times 100 \times \text{換算係數}$$

樣品重量 $W$ 越小，同樣的 $\Delta V$ 造成的影響就越大——這就是為什麼硫酸銨回收標準品只秤 0.05 g 時，讀取誤差的影響會被放大。

In [ ]:
delta_V = 0.05  # mL, assumed titration reading error

sensitivity_rows = []
for item_name, factor_name in [
    ("milk_powder_blind", "dairy"),
    ("high_gluten_flour", "wheat"),
    ("ammonium_sulfate_std", None),  # report as %N, not protein%
]:
    row = df[df["item"] == item_name].iloc[0]
    W_mg = row["weight_g"] * 1000
    delta_pctN = delta_V * N_HCL * 14.007 / W_mg * 100
    if factor_name is not None:
        delta_val = delta_pctN * FACTORS[factor_name]
        label, unit = "蛋白質%", "protein pct-points"
    else:
        delta_val = delta_pctN
        label, unit = "%N", "%N pct-points"
    sensitivity_rows.append({
        "item": item_name, "weight_g": row["weight_g"],
        "delta_V_mL": delta_V, "delta_pctN": round(delta_pctN, 4),
        f"impact ({label})": round(delta_val, 4),
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df

In [ ]:
print("結論：樣品重量越小（如硫酸銨標準品只秤 0.05 g），"
      "同樣 0.05 mL 的讀取誤差，換算成 %N 或蛋白質% 的影響就越大。")
print("這也是為什麼 QC 標準品的秤重與滴定終點判讀，需要格外小心。")

## 8. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果忘記扣除試劑空白（直接用 $V_s$ 計算 %N，不減去 $V_b$），全脂奶粉盲樣班級平均的蛋白質% 會被高估多少（百分點與相對%）？請用 `milk` 資料實際計算比較。

In [ ]:
# TODO: 請在這裡作答

### 問題 2

如果某組不小心把高筋麵粉的 %N 誤用一般係數 6.25（而非正確的小麥係數 5.7），算出的蛋白質% 與相對誤差是多少？跟第 4 節算出的 ~9.6% 是否一致？

In [ ]:
# TODO: 請在這裡作答

### 問題 3

蒸餾回收標準品（硫酸銨）的回收率如果只有 90%，該懷疑哪個步驟出了問題？（不需要計算，寫出你的推理：消化、蒸餾、還是滴定？為什麼？）

In [ ]:
# TODO: 請在這裡作答

### 問題 4

承接三聚氰胺案例：如果某奶粉只摻入 0.1% 的三聚氰胺（比第 6 節示範的量少很多），用乳品係數 6.38 換算，表觀蛋白質會虛增多少百分點？這樣的虛增量，你覺得容不容易被發現？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

利用第 7 節的靈敏度分析：如果你們這組的滴定讀取誤差其實是 0.10 mL（而不是範例的 0.05 mL），對全脂奶粉與高筋麵粉的蛋白質% 分別造成多少影響？兩者受影響的程度誰比較大？為什麼？

In [ ]:
# TODO: 請在這裡作答

## 9. 匯出結果 (Export)

把本週計算結果（全脂奶粉盲樣班級數據＋兩種麵粉＋回收率）存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載。

In [ ]:
export_rows = milk[["group", "student", "value", "weight_g", "pctN", "protein_pct"]].copy()
export_rows["item"] = "milk_powder_blind"

summary_row = pd.DataFrame([{
    "group": "class_mean_n5", "student": "", "value": np.nan, "weight_g": np.nan,
    "pctN": milk["pctN"].mean(), "protein_pct": mean_protein, "item": "milk_powder_blind_summary",
}])

flour_rows = pd.DataFrame([
    {"group": hg["group"], "student": hg["student"], "value": hg["value"],
     "weight_g": hg["weight_g"], "pctN": hg_pctN, "protein_pct": hg_protein_correct,
     "item": "high_gluten_flour"},
    {"group": lg["group"], "student": lg["student"], "value": lg["value"],
     "weight_g": lg["weight_g"], "pctN": lg_pctN, "protein_pct": lg_protein,
     "item": "low_gluten_flour"},
])

results_df = pd.concat([export_rows, summary_row, flour_rows], ignore_index=True)

output_path = "nb05_student_results.csv"
results_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")